# The Four Minds — building an app with `AppSpec`

This notebook shows how a host app plugs its own **domain** into the dialectical
framework with a single declarative object: an **`AppSpec`**. The theme is the
four faculties of the inner instrument (*antahkarana*) — Manas, Buddhi,
Ahamkara, Chitta.

An app is made of up to four pieces, all optional:

| Piece | What it is | Which heads use it |
|---|---|---|
| `voicing` | Domain flavor layered on the Navigator contract | Analyst, Explorer, counsel toggle |
| `advisor_persona` | The entire identity of the *standalone* Advisor (machinery hidden) | standalone Advisor only |
| `tool_guide` | Usage rules for the app's tools, repeated verbatim into every head | all heads |
| `tools` | App-provided `@llm.tool` functions | all heads |

You write ONE `AppSpec` and pass it to every agent head (`Analyst(app=...)`,
`Explorer(app=...)`, `Advisor(app=...)`); each head composes the right preamble
from it. Without AppSpec you'd need framework lore — which base constant belongs
to which head, section override order, repeating the tool guide everywhere.

## Prerequisites

1. **Start Memgraph** (agents build a graph behind the scenes):
   `docker compose -f docker-compose.test.yml up -d` (or `/df-memgraph start`).
2. **Configure the LLM**: copy `.env.example` to `.env` and set
   `DIALEXITY_DEFAULT_MODEL` (e.g. `bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0`)
   plus the matching provider credentials (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, or AWS creds).
3. **Run the cells top to bottom.** Jupyter supports top-level `await`.

## 1. Bootstrap

`DialecticalReasoning.setup(...)` builds and auto-wires the DI container (the one
line a host app runs at startup). A committed `Case` owns the `sid` scope that
every graph write is bound to.

In [ ]:
from dialectical_framework.dialectical_reasoning import DialecticalReasoning
from dialectical_framework.settings import Settings
from dialectical_framework.graph.nodes.case import Case
from dialectical_framework.graph.scope_context import scope
from dialectical_framework.agents.app_spec import AppSpec
from dialectical_framework.agents.advisor.advisor import Advisor

# Build + auto-wire the DI container once (reads .env via Settings.from_env()).
DialecticalReasoning.setup(Settings.from_env())

# A Case owns the sid scope that all graph writes are bound to.
case = Case()
case.commit()
print(f"Case ready — sid={case.sid}")

## 2. A domain tool: the faculty reference

Apps bring field knowledge two ways: **prose** (in the spec's text pieces) and
**callable resources** (as `@llm.tool` functions in `tools`). Here the resource is
a tiny canonical reference for the four faculties — in a real app this could be a
knowledge-base fetch, a chart lookup, anything.

The tool's schema (name, params, docstring) reaches the LLM automatically through
the tool protocol; *when to reach for it* is what `tool_guide` is for.

In [ ]:
from typing import Annotated
from mirascope import llm
from pydantic import Field

FACULTIES = {
    "manas": (
        "Manas — the sensing, impulsive mind. Registers experience and wants to act on it "
        "now. Strength: aliveness, contact with what is actually felt. Blindspot: mistakes "
        "urgency for truth."
    ),
    "buddhi": (
        "Buddhi — discernment. Weighs, compares, sees consequences. Strength: clarity and "
        "judgment. Blindspot: can talk the person out of every risk, including the necessary ones."
    ),
    "ahamkara": (
        "Ahamkara — the 'I'-maker. Maintains identity and continuity: who am I if this changes? "
        "Strength: coherence and self-respect. Blindspot: defends the existing self-image even "
        "against growth."
    ),
    "chitta": (
        "Chitta — memory and conditioning. Holds everything lived through and pattern-matches "
        "the present to it. Strength: continuity of learning. Blindspot: replays old outcomes "
        "as if they were predictions."
    ),
}


@llm.tool
async def faculty_reference(
    faculty: Annotated[
        str, Field(description="One of: manas, buddhi, ahamkara, chitta")
    ],
) -> str:
    """Canonical description of one of the four inner faculties (antahkarana)."""
    key = faculty.lower().strip()
    return FACULTIES.get(
        key, f"Unknown faculty: {faculty!r}. Valid: {', '.join(FACULTIES)}"
    )

## 3. Declare the app

One `AppSpec`, all four pieces. Note the division of labor:

- `advisor_persona` is a full identity — the standalone Advisor hides the
  machinery, so this text is everything the user experiences.
- `voicing` is only a *flavor* — the Navigator heads (Analyst/Explorer) keep their
  own transparent-framework contract and layer this on top.
- Neither piece mentions dialectics or tools mechanics — that's the engine's job.

In [ ]:
FOUR_MINDS_APP = AppSpec(
    voicing="""## Four-Minds Voicing

Where natural, frame tensions in terms of the four inner faculties: Manas
(impulse), Buddhi (discernment), Ahamkara (identity), Chitta (conditioning).
An opposition is often two faculties each holding a truth the other cannot see.
Use plain language first, the Sanskrit name second.
""",
    advisor_persona="""## Persona

You are a contemplative guide for inner work. People come to you pulled between
competing inner voices — the impulse that wants to act, the judgment that weighs,
the identity that needs to protect itself, the memory that keeps replaying. You
help them sit with these voices rather than pick a winner too quickly.

You listen for the voice that is NOT being spoken. When someone leads with one
faculty — raw impulse, cold analysis, wounded ego, old conditioning — you gently
name what the other faculties can see that this one cannot. You frame each blindspot
as something the person already carries within, not a flaw to fix.

When you offer a way forward, you offer it as a practice to try and something to
notice while doing it — never a verdict. Your tone is calm, spacious, and unhurried;
plain, grounded language, no spiritual jargon.
""",
    tool_guide="""## Faculty Reference

- faculty_reference(faculty): canonical description of manas / buddhi / ahamkara /
  chitta — strengths and blindspots. Consult it before characterizing a faculty,
  so the vocabulary stays consistent across the conversation.
""",
    tools=[faculty_reference],
)

## 4. See what each head composes

The spec is declarative; composition is per head. The standalone Advisor gets
`advisor_persona + tool_guide` (no Navigator base — machinery hidden). The
Navigator heads get `NAVIGATOR_APP + voicing + tool_guide`.

In [ ]:
standalone = FOUR_MINDS_APP.advisor_preamble(scoped=False)
navigator = FOUR_MINDS_APP.navigator_preamble()

print("— standalone Advisor preamble —")
print(standalone[:400], "...")
print()
print(f"— Navigator preamble ({len(navigator)} chars; starts with the Navigator contract) —")
print(navigator[:400], "...")

## 5. Chat with the standalone Advisor

`Advisor(app=FOUR_MINDS_APP)` — the spec supplies persona, tool guide, and the
`faculty_reference` tool in one argument. All chat happens inside
`with scope(case.sid):` so graph writes land in this Case. History is retained on
`advisor.messages`.

In [ ]:
with scope(case.sid):
    advisor = Advisor(app=FOUR_MINDS_APP)
    reply = await advisor.chat(
        "Part of me is desperate to quit my job and start over, and part of me "
        "says that's reckless and I'd regret throwing away ten years. I keep "
        "circling and can't tell which voice to trust."
    )
print(reply)

In [ ]:
# Follow-up turn — same scope, same advisor, history preserved.
with scope(case.sid):
    reply = await advisor.chat(
        "The reckless voice is the loud one. But underneath it I think I'm just "
        "exhausted and want to feel something again."
    )
print(reply)

## 6. The same spec drives every head

That's the point of `AppSpec`: define the app once, hand it to whichever head the
session needs.

```python
from dialectical_framework.agents.analyst.analyst import Analyst
from dialectical_framework.agents.explorer.explorer import Explorer

with scope(case.sid):
    analyst = Analyst(app=FOUR_MINDS_APP)          # transparent Navigator, four-minds voicing
    # ... after the Analyst creates a nexus:
    explorer = Explorer(nexus_hash=nx, app=FOUR_MINDS_APP)
    # counsel toggle of that same exploration (same messages, same nexus):
    advisor = Advisor(nexus_hash=nx, messages=explorer.messages, app=FOUR_MINDS_APP)
```

(The `seasons.ipynb` notebook next door runs the Analyst path live.)

**Try your own:**
- Change a single piece — e.g. drop `advisor_persona` and the standalone Advisor
  falls back to the bare engine; drop everything but `voicing` for a pure re-skin.
- Swap in a shipped persona instead: `from dialectical_framework.agents.apps import COACH_PERSONA`
  and `Advisor(app_preamble=COACH_PERSONA)` (the manual layer — `app=` and
  `app_preamble=` are mutually exclusive).
- Inspect the conversation with `advisor.messages`; resume later via
  `Advisor(app=FOUR_MINDS_APP, messages=saved)`.